### **서울 열린데이터 광장 활용**
>[미세먼지 데이터](https://data.seoul.go.kr/dataList/OA-2219/S/1/datasetView.do)

>[API](http://openapi.seoul.go.kr:8088/***REMOVED_SEOUL_KEY***/xml/RealtimeCityAir/1/5/)

#### **실습 1) API로부터 데이터 받아오기**

In [ ]:
from http.client import HTTPConnection

hc = HTTPConnection("openapi.seoul.go.kr", 8088) # '대상 폴더' 직전까지의 도메인, 또는 ip, 포트 등

# hc.request("GET", "/***REMOVED_SEOUL_KEY***/xml/RealtimeCityAir/1/5/") 
# 마지막의 /1/5/ 구간 입력값을 변경해, 원하는 권역을 볼 수 있다.
hc.request("GET", "/***REMOVED_SEOUL_KEY***/xml/RealtimeCityAir/1/25/")

res = hc.getresponse()
resBody = res.read()
print(resBody.decode())

hc.close()

#### **XML: eXtended Markup Language**
>HTML 형태로 표현된, DOM(Document Object Model) 데이터 객체
>>**```<태그명 속성 = "값"> :```**

#### **실습 2: 데이터 출력 및 저장**

1. API 호출
2. 데이터 추출
3. 기존 csv를 확인해 불러오고, 문자열로 변환해 비교 - 중복 데이터가 추가되는 것을 방지.
4. 신규 데이터 필터링.
5. 신규 데이터를 추가해 저장. 

In [3]:
import xml.etree.ElementTree as ET
from http.client import HTTPConnection
import pandas as pd
import os
import datetime 

# 1. API 통신 및 resBody 획득
host = "openapi.seoul.go.kr"
port = 8088
url_path = "/***REMOVED_SEOUL_KEY***/xml/RealtimeCityAir/1/25/" 
resBody = None

try:
    hc = HTTPConnection(host, port)
    hc.request("GET", url_path)
    res = hc.getresponse()
    
    if res.status == 200:
        resBody = res.read()
    else:
        print(f"오류 발생: HTTP 상태 코드 {res.status}")
        print(res.read().decode())
        
    hc.close()

except Exception as e:
    print(f"통신 중 오류 발생: {e}")

# 2. XML 파싱 및 데이터 추출
data_list = []
if resBody:
    seoulDustData = ET.fromstring(resBody) 
    row_s = seoulDustData.iter("row")
    
    for r in row_s:
        msrmt_raw = r.find('MSRMT_DT').text if r.find('MSRMT_DT') is not None else 'N/A'
                
        extracted_data = {
            '측정일시': msrmt_raw, 
            '권역명': r.find('SAREA_NM').text if r.find('SAREA_NM') is not None else 'N/A',
            '측정소명': r.find('MSRSTN_NM').text if r.find('MSRSTN_NM') is not None else 'N/A',
            'PM': r.find('PM').text if r.find('PM') is not None else 'N/A',
            'FPM': r.find('FPM').text if r.find('FPM') is not None else 'N/A',
            '통합등급': r.find('CAI_IDX').text if r.find('CAI_IDX') is not None else 'N/A'
        }
        data_list.append(extracted_data)

# 3. 중복 제거 및 CSV 저장
if data_list:
    df_new = pd.DataFrame(data_list)
    
    csv_file_name = "seoul_air_quality_data.csv"
    key_cols = ['측정일시', '측정소명']
    
    if os.path.isfile(csv_file_name):
        # 3-1. 파일이 존재하면 기존 데이터 로드
        df_old = pd.read_csv(csv_file_name, encoding='utf-8-sig')
        
        # 기존 데이터의 키 컬럼을 문자열로 통일 (타입 불일치 문제 해결)
        df_old[key_cols] = df_old[key_cols].astype(str)
        
        # 3-2. 기존 데이터에 포함되지 않은 '새로운' 행만 필터링
        old_keys = set(df_old.apply(lambda row: tuple(row[col] for col in key_cols), axis=1))
        
        is_new = df_new.apply(lambda row: tuple(row[col] for col in key_cols) not in old_keys, axis=1)
        df_to_save = df_new[is_new]
        
        print(f"\n기존 파일에 {len(df_old)}개 레코드 존재.")
        print(f"새로 가져온 {len(df_new)}개 레코드 중, {len(df_to_save)}개 레코드만 신규 데이터입니다.")
        
    else:
        df_to_save = df_new
    
    
    # 4. 신규 데이터만 CSV에 추가 저장
    if not df_to_save.empty:
        file_exists = os.path.isfile(csv_file_name) 
        
        df_to_save.to_csv(
            csv_file_name, 
            mode='a', 
            header=not file_exists, 
            index=False, 
            encoding='utf-8-sig'
        )
        print(f"\n--- 저장 완료 ---")
        print(f"'{csv_file_name}' 파일에 {len(df_to_save)}개의 신규 레코드를 추가했습니다.")
    else:
        print("\n--- 저장 완료 ---")
        print("기존 파일에 이미 존재하는 데이터이므로 추가된 레코드가 없습니다.")


기존 파일에 100개 레코드 존재.
새로 가져온 25개 레코드 중, 25개 레코드만 신규 데이터입니다.

--- 저장 완료 ---
'seoul_air_quality_data.csv' 파일에 25개의 신규 레코드를 추가했습니다.
